# Classroom Vision · 실행

**전체 실행**을 누르면 외부 Python 실행 준비 창이 열립니다.
카메라를 **이름**으로 고르거나 영상 파일을 선택하고 **추적 시작**을 누르세요.
기존 사람 추적 UI·이름·인원 집계·자리 비움 기능은 그대로 사용합니다.

로컬 Python 3.13 커널을 권장합니다. CUDA → Mac MPS → CPU 자동 선택과 직접 선택을 지원합니다.
최초 설치 후에는 모델을 보관해 인터넷 없이 실행할 수 있습니다.
VS Code 위젯 확장이나 외부 CDN은 사용하지 않습니다.

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import os
import subprocess
import sys

print("[1/3] 실행 환경을 확인합니다…", flush=True)
if sys.version_info < (3, 11):
    raise RuntimeError("로컬 Python 3.13 커널을 선택해 주세요.")
DEPLOY = next((p.resolve() for p in (Path.cwd(), Path.cwd() / "deploy")
               if (p / "launcher.py").is_file() and (p / "models/manifest.json").is_file()), None)
if DEPLOY is None:
    raise RuntimeError("deploy 폴더의 START.ipynb를 열어 주세요.")
from packaging.requirements import Requirement
requirements = [Requirement(line.strip()) for line in (DEPLOY / "requirements.txt").read_text().splitlines()
                if line.strip() and not line.startswith("#")]
missing = []
for requirement in requirements:
    try:
        if not requirement.specifier.contains(metadata.version(requirement.name)):
            missing.append(requirement.name)
    except metadata.PackageNotFoundError:
        missing.append(requirement.name)


def run_visible(command):
    environment = dict(os.environ, PYTHONUNBUFFERED="1", PIP_PROGRESS_BAR="off")
    with subprocess.Popen(command, cwd=DEPLOY, env=environment, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            if not line.startswith("Requirement already satisfied:"):
                print(line.rstrip(), flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError("준비 과정에 문제가 있습니다. 위 메시지를 확인해 주세요.")

if missing:
    print("최초 패키지 준비: " + ", ".join(missing), flush=True)
    run_visible([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                 "--progress-bar", "off", "-r", str(DEPLOY / "requirements.txt")])
print("[2/3] 파인튜닝 모델과 보조 모델을 확인합니다…", flush=True)
run_visible([sys.executable, "-u", str(DEPLOY / "models.py"), "--prepare"])
print("실행 환경과 모델 준비 완료", flush=True)

### 실행 준비 창 열기

아래 셀은 바로 끝나며, 카메라·영상·장치 선택은 **외부 실행 준비 창**에서 합니다.
카메라 연결을 바꿨다면 **새로고침**을 누르세요.

In [ ]:
print("[3/3] 외부 실행 준비 창을 엽니다…", flush=True)
process = globals().get("_classroom_launcher")
if process is not None and process.poll() is None:
    print("실행 준비 창이 이미 열려 있습니다. 해당 창에서 추적 시작을 눌러 주세요.", flush=True)
else:
    (DEPLOY / "outputs").mkdir(exist_ok=True)
    _launcher_log = DEPLOY / "outputs" / f"launcher-{os.getpid()}.log"
    with _launcher_log.open("w", encoding="utf-8") as log:
        options = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if os.name == "nt" else {"start_new_session": True}
        _classroom_launcher = subprocess.Popen([sys.executable, "-u", str(DEPLOY / "launcher.py")],
                                               cwd=DEPLOY, stdout=log, stderr=subprocess.STDOUT,
                                               env=dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUTF8="1"), **options)
    print("실행 준비 창을 시작했습니다. 창이 열리지 않으면 아래 확인 셀을 실행하세요.", flush=True)

### 창이 열리지 않을 때만 실행

In [ ]:
process = globals().get("_classroom_launcher")
if process is None:
    print("위의 실행 준비 창 열기 셀을 먼저 실행해 주세요.")
elif process.poll() is None:
    print("실행 준비 창이 실행 중입니다. 작업 표시줄/Dock에서 Python 창을 확인하세요.")
else:
    print("실행 준비 창이 종료됐습니다. 다시 열려면 위 셀을 실행하세요.")
    log = globals().get("_launcher_log", DEPLOY / "outputs/launcher.log")
    if log.exists():
        print(log.read_text(encoding="utf-8", errors="replace")[-4000:])